Script: Plot NEXRAD LevelII .ar2v binary file  
Author: Van  
Version: 20230815  
Path: D:\Seafile\Share\notebooks\20230601ReadFile\NEXRAD_leve2_ar2v_plot.ipynb

# 1.READ

### 1.1 using level2

In [ ]:
import pyart.io.nexrad_level2 as ar2v
from pyart.core.transforms import antenna_vectors_to_cartesian
import numpy as np

indir='D:/Seafile/Share/notebooks/20230523DisplayGUI/dual_pol_radar_plot/testdata/'
fname='KFFC20210605_000151_V06'
fname='KOUN20120107_000322_V06'
indir='./'
fname='NJU.20220729.105032.AR2.bz2.ar2v'
fname='Z_RADR_I_Z9791_20230526071116_O_DOR_SAD_CAP_FMT.bin.bz2.ar2v'
fname='HEBR.20210601.084654.ar2v.ar2v'
inpath=indir+fname
ar2v_radar=ar2v.NEXRADLevel2File(inpath)
nswps=ar2v_radar.nscans
print(nswps)
swp=0
field,field_name,field_num='dBZ','REF',0
scan_info=ar2v_radar.scan_info()
scan_msgs=ar2v_radar.scan_msgs
nrays=scan_info[swp]['nrays']
ngates=scan_info[swp]['ngates'][field_num]
gate_spacing=scan_info[swp]['gate_spacing'][field_num]

field_data=ar2v_radar.get_data(field_name,scan_info[swp]['ngates'][field_num],[swp])
r=ar2v_radar.get_range(scan_num=swp,moment=field_name)
azi=ar2v_radar.get_azimuth_angles(scans=[swp])
ele=ar2v_radar.get_elevation_angles(scans=[swp])
x,y,z=antenna_vectors_to_cartesian(r,azi,ele)
xx,yy,cc=x/1000,y/1000,field_data

In [ ]:
ar2v_radar._records[0]

### 1.2 a using pyart radar only

In [ ]:
import pyart
radar=pyart.io.read(inpath)
print(radar.sweep_number['data'])
field_data=radar.get_field(swp,'reflectivity')
x,y,z=radar.get_gate_x_y_z(swp)
xx,yy,cc=x/1000,y/1000,field_data

### 1.2 b using pyart graph and plot manually

In [ ]:
rp = pyart.graph.RadarDisplay(radar)
cc = rp._get_data('reflectivity', swp, mask_tuple=None, filter_transitions=True, gatefilter=None)
xx,yy,zz=radar.get_gate_x_y_z(swp)
xx,yy=xx/1000,yy/1000

### 1.3 using pyart graph and plot automatically

In [ ]:
import pyart
radar=pyart.io.read(inpath)
rp = pyart.graph.RadarDisplay(radar)
rp.plot_ppi('reflectivity',swp)

# 2.PLOT

In [ ]:
################################################################################################
# functions
def radarCmap(field):
    # match different radar field variables to different colormaps
    field_map={'dBZ':'reflectivity',
          'KDP':'specific_differential_phase',
          'ZDR':'differential_reflectivity',
          'VEL':'velocity',
          'rhohv':'cross_correlation_ratio'}
    # print(field_map[field])
    from matplotlib import cm,colormaps
    from matplotlib.colors import ListedColormap,BoundaryNorm
    import numpy as np
    import cmaps
    # import cmaps
    if field_map[field]==field_map['dBZ']:
        c1=[255,255,255,256]
        c2=[0,236,236,256]
        c3=[1,160,246,256]
        c4=[0,0,246,256]
        c5=[0,255,0,256]
        c6=[0,200,0,256]
        c7=[0,144,0,256]
        c8=[255,255,0,256]
        c9=[231,192,0,256]
        c10=[255,144,0,256]
        c11=[255,0,0,256]
        c12=[214,0,0,256]
        c13=[192,0,0,256]
        c14=[221,94,253,256]
        c15=[191,2,238,256]
        c16=[85,1,106,256]
        c=[c1,c2,c3,c4,c5,c6,c7,c8,c9,c10,c11,c12,c13,c14,c15,c16]
        cmap = ListedColormap(np.array(c)/256)
        norm= BoundaryNorm(field_levs[field],cmap.N)
    elif field_map[field]==field_map['rhohv']:
        # cmap = cm.jet
        try:
            cmap=cmaps.Carbone42
        except:
            cmap=colormaps['Carbone42']
        colorslist=cmap(np.linspace(0,1,len(field_levs[field])-1)) #cmap(np.exp(np.linspace(0,2,len(field_levs[field])-1))/np.exp(2))
        cmap = ListedColormap(colorslist)
        norm = BoundaryNorm(field_levs[field],cmap.N)
        # cmap,norm= from_levels_and_colors(field_lev[field],colorslist)
    else:
        try:
            cmap=cmaps.Carbone42
        except:
            cmap=colormaps['Carbone42']
        colorslist=cmap(np.linspace(0,1,len(field_levs[field])-1)) 
        cmap = ListedColormap(colorslist)
        norm = BoundaryNorm(field_levs[field],cmap.N)
    return cmap,norm

def perfect_levs(levs):
    # transfer levs(list or array) to a new list
    # to draw beatiful ticklabels, like ' 0 0.1 1' instead of '0.0 0.1 1.0' with ugly '.0' 
    levs_list=list(levs)
    for i in range(len(levs)):
        if levs[i]%1==0: # is integer
            levs_list[i]=int(levs[i])
    return levs_list

# new dicts
import numpy as np
field_map={'dBZ':'reflectivity',
          'KDP':'specific_differential_phase',
          'ZDR':'differential_reflectivity',
        #   'VEL':'velocity',
          'rhohv':'cross_correlation_ratio'}
field_lim={'dBZ':[-10,70],
           'KDP':[-2,5],
           'ZDR':[-1,8],
           'VEL':[-30,30],
           'rhohv':[0,1]}
field_levs={'dBZ':perfect_levs(np.linspace(-10,70,17)),
           'KDP':perfect_levs(np.linspace(-2,5,15)),
           'ZDR':perfect_levs(np.linspace(-1,8,19)),
           'VEL':perfect_levs(np.linspace(-30,30,13)),
           'rhohv':[0,.1,.3,.5,.6,.7,.8,.85,.9,.92,.94,.95,.96,.97,.98,.99,1]}
field_cmap={'dBZ':radarCmap('dBZ'),
           'KDP':radarCmap('KDP'),
           'ZDR':radarCmap('ZDR'),
           'VEL':radarCmap('VEL'),
           'rhohv':radarCmap('rhohv')}
field_locs={'dBZ':field_levs['dBZ'],
           'KDP':field_levs['KDP'],
           'ZDR':field_levs['ZDR'],
           'VEL':field_levs['VEL'],
           'rhohv':np.linspace(0,1,len(field_levs['rhohv']))}
################################################################################################

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colorbar import ColorbarBase
# savename=str(swp+1)+'_swp='+str(sweep[swp])+'deg_'+field+'.png'

plt.close()
fig,ax1=plt.figure(figsize=(8, 7)),plt.subplot(111)
im=plt.pcolormesh(xx,yy,cc,cmap=field_cmap[field][0],norm=field_cmap[field][1]) #vmin=field_lim[field][0],vmax=field_lim[field][1])
# im=plt.contourf(xx,yy,cc,cmap=field_cmap[field][0],norm=field_cmap[field][1],levels=field_levs[field]) 
cax = fig.add_axes([ax1.get_position().x1+0.01,ax1.get_position().y0,0.02,ax1.get_position().y1-ax1.get_position().y0])   
cb=ColorbarBase(cax,cmap=field_cmap[field][0],norm=field_cmap[field][1],ticks=field_levs[field])
# plt.savefig(outpath+savename, dpi=600)